# Microblogging · TinyFlux

[Demo-Übersicht](README.md) · [Vorbereitung](../../README_technical_preparation.md) · [UAP-Aufgabe](../../tasks/04_tinyflux/README.md)

Wie verändert sich die Aktivität einer Plattform über die Zeit? **Punktmodell, Zeitfenster und Q4 bilden den Kern.** Q1–Q3 und Q5 bleiben ausführbare Vergleichsfragen; DAU ergänzt das ursprüngliche Beispiel.

TinyFlux speichert hier einzelne Ereignisse in einer lokalen CSV-basierten Datenbank. Zeit- und Tag-Filter führt TinyFlux aus; Gruppierung, Tabellenverknüpfung, Pfadsuche und gleitende Berechnung erfolgen ausdrücklich in Python beziehungsweise pandas. Das Beispiel behauptet keine verteilte Zeitreihenplattform.

## 0. Umgebung und vollständige Eingaben

Kernel: **Python (rothstein-storage-workshop-2026)**. TinyFlux 1.2.0 ist bereits in der Kursumgebung enthalten; ein Datenbankserver wird für dieses Notebook nicht benötigt.

Wir verwenden die vollständigen relationalen CSV-Eingaben, ohne eine SQLite-Datenbank vorauszusetzen. Die alte `daily_activity.csv` enthält nur 30 Kalendertage bis einschliesslich 10. September 2025. Likes und Kommentare reichen in den Detaildaten darüber hinaus. Am Ende vergleichen wir diese Abdeckung; die Originaldateien bleiben unverändert.

Zeitzonenlose Zeitangaben der synthetischen Plattformdaten werden im Kurs als UTC interpretiert. Bei Follow-Daten mit Tagespräzision dient 00:00 UTC als technische Tagesmarke, nicht als nachgewiesene Uhrzeit. Alle Zeitfenster sind fest an den Bestand gebunden.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display
from tinyflux import Point, TinyFlux, TimeQuery, TagQuery, FieldQuery

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/tinyflux_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from tinyflux_workshop import (csv_rows, parse_time, database_path, prepare_points,
    replace_snapshot, search_points, points_frame, annual_inputs, agency_names,
    yearly_frame, daily_posts)
Time, Tag, Field = TimeQuery(), TagQuery(), FieldQuery()
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.figsize": (10, 4.3), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATASET = "microblogging"
points = prepare_points(DATASET)
print("Ereignisse:", len(points))
print("Arbeitsdatei:", database_path(DATASET))
print(next(p for p in points if p.tags["type"] == "comment"))

## 1. Ein Punkt im Modell

| Bestandteil | Verwendung |
| --- | --- |
| `time` | Zeitpunkt des Ereignisses; bei Follows technische Tagesmarke |
| `measurement` | `microblogging_events` |
| `tags` | Ereignistyp, Nutzer-ID, Ereignis-ID und Post-ID beziehungsweise Zielnutzer |
| `fields` | Numerischer Zählwert `value=1` |

Die IDs sind hier Strings in Tags. Zwei Kommentare derselben Person zum selben Post bleiben dank unterschiedlicher `event_id` zwei Ereignisse. Ein gemeinsamer Zeitstempel ist **kein eindeutiger Schlüssel**. Viele ID-Tags sind für diesen kleinen Lehrbestand praktisch; ihre hohe Kardinalität müsste bei der Wahl eines grossen Zeitreihensystems bewusst bewertet werden.

Der folgende Import ersetzt nur die Demo-Arbeitsdatei. Er schreibt zunächst eine temporäre Datei und prüft deren Inhalt nach erneutem Öffnen. Wiederholungen erzeugen denselben Bestand. Das ist Verhalten unseres Importcodes; TinyFlux selbst erzwingt hier keine Eindeutigkeit. Eigene Änderungen in dieser Demo-Datei werden beim Neuimport zurückgesetzt.

In [ ]:
first_import = replace_snapshot(DATASET, points=points)
second_import = replace_snapshot(DATASET, points=points)
assert first_import == second_import
all_events = points_frame(search_points(DATASET))
event_counts = all_events.groupby("type")["value"].sum().astype(int)
assert event_counts.to_dict() == {"comment": 512, "follow": 3047, "like": 1266, "post": 865}
display(event_counts.rename("Ereignisse").to_frame())
print("IMPORT OK – nach Wiederholung weiterhin", len(all_events), "Punkte")

## 2. Ein Zeitfenster abfragen

Die Untergrenze ist eingeschlossen, die Obergrenze ausgeschlossen: `[8. September, 11. September 2025)`. Benachbarte Zeitfenster können so ohne doppelte Randpunkte aneinander anschliessen. Klammern und `&` verbinden die TinyFlux-Bedingungen.

In [ ]:
window_start = datetime(2025, 9, 8, tzinfo=timezone.utc)
window_end = datetime(2025, 9, 11, tzinfo=timezone.utc)
window_query = (Time >= window_start) & (Time < window_end) & (Tag.type == "post")
window_posts = points_frame(search_points(DATASET, window_query))
display(window_posts[["time", "user_id", "post_id", "value"]].head())
print("Posts im Fenster:", len(window_posts))

## Q4 · Posts pro Kalendertag

TinyFlux liefert die Post-Ereignisse. `daily_posts` summiert sie danach in pandas pro UTC-Kalendertag, ergänzt fehlende Tage innerhalb des Postzeitraums mit Null und berechnet den gleitenden Durchschnitt. Die ersten sechs Werte verwenden die bis dahin verfügbaren Tage.

Sieben Zeilen repräsentieren nach der Kalenderergänzung sieben aufeinanderfolgende Tage. Ein gleitender Durchschnitt über unregelmässige Ereigniszeilen hätte eine andere Bedeutung.

In [ ]:
post_events = points_frame(search_points(DATASET, Tag.type == "post"))
q4 = daily_posts(post_events)
assert len(q4) == 30 and int(q4["posts"].sum()) == 865
fig, ax = plt.subplots(layout="constrained")
ax.bar(q4["day"], q4["posts"], color="#87b9c7", label="Posts pro Kalendertag")
ax.plot(q4["day"], q4["rolling_7d"], color="#193c5a", linewidth=2.5,
        label="Durchschnitt über bis zu 7 Tage")
ax.set(title="Microblogging: Verlauf der veröffentlichten Posts", xlabel="Kalendertag (UTC)",
       ylabel="Anzahl Posts")
ax.legend(frameon=False)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()
display(q4.head(8))

## Q1 · Aktivität pro Person — Vertiefung

Aktivität = verfasste Posts + verfasste Kommentare + **gegebene** Likes über den gesamten Eingabebestand. Follow-Ereignisse zählen nicht dazu. Die Nutzerliste ergänzt auch Konten ohne Aktivität. TinyFlux filtert; pandas gruppiert und verknüpft.

In [ ]:
users = pd.DataFrame(csv_rows("data/microblogging/relational/users.csv"))
users["user_id"] = users["user_id"].astype(int)
active_points = search_points(DATASET, (Tag.type == "post") | (Tag.type == "comment") | (Tag.type == "like"))
activity = points_frame(active_points)
activity["user_id"] = activity["user_id"].astype(int)
counts = activity.pivot_table(index="user_id", columns="type", values="value", aggfunc="sum", fill_value=0)
q1 = users[["user_id", "username"]].merge(counts, on="user_id", how="left").fillna(0)
q1[["post", "comment", "like"]] = q1[["post", "comment", "like"]].astype(int)
q1["engagement_score"] = q1[["post", "comment", "like"]].sum(axis=1)
q1 = q1.sort_values(["engagement_score", "user_id"], ascending=[False, True]).reset_index(drop=True)
assert q1.head(3)[["user_id", "engagement_score"]].values.tolist() == [[94, 26], [107, 26], [35, 25]]
display(q1.head(10))

## Q2 · Empfangene Likes pro Post — Vertiefung

Jetzt gruppieren wir Like-Ereignisse nach dem **Post**, nicht nach der gebenden Person. Ein Left Join erhält Posts ohne Likes. Autorname und Text stammen als Stammdaten aus den CSV-Eingaben; TinyFlux übernimmt diese Verknüpfung nicht.

In [ ]:
posts = pd.DataFrame(csv_rows("data/microblogging/relational/posts.csv"))
posts[["post_id", "user_id"]] = posts[["post_id", "user_id"]].astype(int)
posts["created_at"] = pd.to_datetime(posts["created_at"], utc=True)
likes = points_frame(search_points(DATASET, Tag.type == "like"))
likes["post_id"] = likes["post_id"].astype(int)
received = likes.groupby("post_id")["value"].sum().rename("like_count")
q2 = posts.merge(received, on="post_id", how="left").fillna({"like_count": 0})
q2["like_count"] = q2["like_count"].astype(int)
q2 = q2.merge(users[["user_id", "username"]], on="user_id", how="left")
q2 = q2.sort_values(["like_count", "post_id"], ascending=[False, True]).reset_index(drop=True)
assert len(q2) == 865 and q2["like_count"].sum() == 1266
assert (q2["like_count"] == 0).any()
display(q2[["post_id", "username", "like_count", "text"]].head(10))

## Q3 · Follow-Beziehungen — Vertiefung zum Modellvergleich

TinyFlux liefert Follow-Ereignisse. Aus ihnen bauen wir **in Python** die Nachbarschaftsliste und suchen per Breitensuche einen kürzesten gerichteten Pfad von 1 nach 3. Die sortierten Nachbarn machen die Auswahl bei gleich langen Pfaden reproduzierbar. Es gibt keine Unfollow-Historie; Follows werden im Snapshot als weiterhin bestehend behandelt.

Die zusätzliche Programmlogik zeigt, weshalb ein Graphmodell diese Frage direkter ausdrücken kann. Das ist kein Leistungstest.

In [ ]:
from collections import defaultdict, deque
follows = points_frame(search_points(DATASET, Tag.type == "follow"))
follows[["user_id", "target_user_id"]] = follows[["user_id", "target_user_id"]].astype(int)
following_count = follows.groupby("user_id").size().rename("following")
follower_count = follows.groupby("target_user_id").size().rename_axis("user_id").rename("followers")
q3_degrees = users[["user_id", "username"]].merge(following_count, on="user_id", how="left")
q3_degrees = q3_degrees.merge(follower_count, on="user_id", how="left").fillna(0)
q3_degrees[["following", "followers"]] = q3_degrees[["following", "followers"]].astype(int)
neighbours = defaultdict(set)
for row in follows.itertuples():
    neighbours[row.user_id].add(row.target_user_id)
queue, visited, q3_path = deque([[1]]), {1}, None
while queue:
    path = queue.popleft()
    if path[-1] == 3:
        q3_path = path
        break
    for target in sorted(neighbours[path[-1]]):
        if target not in visited:
            visited.add(target)
            queue.append(path + [target])
assert q3_path == [1, 2, 3]
print("Gerichteter Pfad:", q3_path, "– Schritte:", len(q3_path) - 1)
display(q3_degrees.sort_values("user_id").head(10))

## Q5 · Feed für Nutzer 1 — Vertiefung

Wir berücksichtigen nur Personen, denen Nutzer 1 bereits zu Beginn des Fensters folgt. Posts liegen im selben halboffenen Fenster wie oben. Die Feed-Regel zeigt Beiträge dieser Follow-Ziele; eigene Posts werden nicht zusätzlich aufgenommen. Die Likes beziehen sich wie bei Q2 auf den gesamten Bestand. TinyFlux filtert beide Ereignismengen; pandas verbindet sie mit den Postdetails.

In [ ]:
viewer = 1
follow_query = (Tag.type == "follow") & (Tag.user_id == str(viewer)) & (Time <= window_start)
followed = {int(p.tags["target_user_id"]) for p in search_points(DATASET, follow_query)}
feed_points = search_points(DATASET, window_query)
feed_ids = [int(p.tags["post_id"]) for p in feed_points if int(p.tags["user_id"]) in followed]
q5 = q2[q2["post_id"].isin(feed_ids)].sort_values(
    ["created_at", "post_id"], ascending=[False, True]).head(50).reset_index(drop=True)
assert q5["post_id"].tolist() == [23, 399, 204, 396]
display(q5[["post_id", "username", "created_at", "like_count", "text"]])

## Zusatz · DAU und Abdeckung des alten Beispiels

**Daily Active Users (DAU)** zählt unterschiedliche Personen, die an einem Tag posten, kommentieren oder liken. Eine Person mit drei Aktivitäten zählt einmal. Follow-Ereignisse zählen nicht dazu. Aggregation, Kalender und gleitender Durchschnitt entstehen in pandas.

Die Originalübersicht umfasst 30 Tage, die vollständigen Aktivitätsereignisse 32. Eine unterschiedliche Summe ist hier eine Frage der Abdeckung. Sie darf nicht als Unterschied zwischen Speichermodellen interpretiert werden.

In [ ]:
activity["day"] = activity["time"].dt.floor("D")
calendar = pd.date_range(activity["day"].min(), activity["day"].max(), freq="D")
dau = activity.groupby("day")["user_id"].nunique().reindex(calendar, fill_value=0).rename("dau").to_frame()
dau["rolling_7d"] = dau["dau"].rolling(7, min_periods=1).mean()
old_daily = pd.DataFrame(csv_rows("data/microblogging/timeseries/daily_activity.csv"))
coverage = pd.DataFrame({
    "Kennzahl": ["Posts", "Likes", "Kommentare"],
    "Detailereignisse": [int(event_counts[k]) for k in ["post", "like", "comment"]],
    "Alte Tagesübersicht": [old_daily[k].astype(int).sum() for k in ["posts", "likes", "comments"]]
})
coverage["Differenz"] = coverage["Detailereignisse"] - coverage["Alte Tagesübersicht"]
display(coverage)
assert coverage["Differenz"].tolist() == [0, 19, 18]
print("Alte Übersicht:", old_daily["day"].min(), "bis", old_daily["day"].max())
print("Detailaktivität:", calendar.min().date(), "bis", calendar.max().date())
fig, ax = plt.subplots(layout="constrained")
ax.plot(dau.index, dau["dau"], color="#87b9c7", marker=".", label="Aktive Personen pro Tag")
ax.plot(dau.index, dau["rolling_7d"], color="#193c5a", linewidth=2.5, label="Durchschnitt über bis zu 7 Tage")
ax.set(title="Microblogging: täglich aktive Personen", xlabel="Kalendertag (UTC)", ylabel="Anzahl Personen")
ax.legend(frameon=False)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()

## Transfer zum UAP-Workshop

Bei der Plattform hat ein Punkt ein einzelnes Ereignis repräsentiert. Im [vierten Workshop](../../tasks/04_tinyflux/README.md) repräsentiert ein Punkt dagegen eine **bereits aggregierte Jahreszählung je Katalogstelle**. Die Punktanzahl und die Summe der Zählwerte beantworten dort unterschiedliche Fragen.

[Modell und Import](../../schemas/tinyflux/README.md) · [TinyFlux-Projekt und API-Beispiele](https://github.com/citrusvanilla/tinyflux)